# Laboratorio 9 — Etapa 5: Simulación del Álbum Real
**MM3014 Teoría de Probabilidades · Universidad del Valle de Guatemala**

---

## Parámetros reales del álbum (Mundial 2026)

| Parámetro | Valor |
|---|---|
| Número total de estampas | N = 980 |
| Estampas por sobre | S = 7 |
| Precio sobre individual | Q 9.50 |
| Precio caja (104 sobres) | Q 975.00 |

---

## Preguntas formuladas

1. **¿Cuántos sobres se necesitan en promedio para alcanzar distintos hitos del álbum (50%, 75%, 90%, 95% y 100%)?**
2. **Con un presupuesto fijo de Q 500 comprando solo sobres individuales, ¿cuál es la distribución del porcentaje del álbum completado y la probabilidad de superar el 70%?**
3. **¿Es más económico completar el álbum comprando cajas (Q 975/104 sobres) o sobres individuales (Q 9.50 c/u)? ¿Cuánto se ahorra en promedio?**
4. **Con una tasa de intercambio K = 6 (cada 6 repetidas se cambian por 1 nueva), ¿cuánto dinero se ahorra en promedio respecto a no intercambiar?**
5. **¿Cómo varía el número esperado de sobres necesarios para completar el álbum si S cambia entre 5, 6, 7, 8 y 9 estampas por sobre?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy import stats

# ── Parámetros globales ──────────────────────────────────────────────
N = 980          # total de estampas
S = 7            # estampas por sobre
P_SOBRE = 9.50   # precio sobre individual (Q)
P_CAJA  = 975.00 # precio caja 104 sobres (Q)
SOBRES_CAJA = 104
RNG = np.random.default_rng(42)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

---
## Funciones auxiliares

In [ ]:
def simular_album(n=N, s=S, rng=RNG):
    """Simula compra de sobres hasta completar el álbum. Devuelve sobres totales."""
    coleccion = np.zeros(n, dtype=bool)
    sobres = 0
    while not coleccion.all():
        nuevas = rng.integers(0, n, size=s)
        coleccion[nuevas] = True
        sobres += 1
    return sobres


def simular_album_hitos(n=N, s=S, hitos=(0.50, 0.75, 0.90, 0.95, 1.00), rng=RNG):
    """Simula hasta completar el álbum y registra el sobre en que se alcanza cada hito."""
    coleccion = np.zeros(n, dtype=bool)
    sobres = 0
    umbrales = [int(h * n) for h in hitos]
    resultados = [None] * len(hitos)
    pendientes = list(range(len(hitos)))
    while pendientes:
        nuevas = rng.integers(0, n, size=s)
        coleccion[nuevas] = True
        sobres += 1
        total = int(coleccion.sum())
        for idx in list(pendientes):
            if total >= umbrales[idx]:
                resultados[idx] = sobres
                pendientes.remove(idx)
    return resultados


def simular_curva_llenado(n=N, s=S, max_sobres=1200, rng=RNG):
    """Devuelve el número de estampas únicas tras cada sobre abierto (hasta max_sobres)."""
    coleccion = np.zeros(n, dtype=bool)
    progreso = []
    for _ in range(max_sobres):
        coleccion[rng.integers(0, n, size=s)] = True
        progreso.append(int(coleccion.sum()))
        if coleccion.all():
            break
    return np.array(progreso)


def simular_con_intercambio(n=N, s=S, k=6, rng=RNG):
    """Simula hasta completar el álbum con intercambio: acumula k repetidas → 1 nueva."""
    coleccion = np.zeros(n, dtype=bool)
    repetidas = 0
    sobres = 0
    while not coleccion.all():
        nuevas = rng.integers(0, n, size=s)
        sobres += 1
        for e in nuevas:
            if coleccion[e]:
                repetidas += 1
            else:
                coleccion[e] = True
        while repetidas >= k and not coleccion.all():
            repetidas -= k
            faltantes = np.where(~coleccion)[0]
            coleccion[rng.choice(faltantes)] = True
    return sobres


def ci95(data):
    """Intervalo de confianza 95% para la media."""
    n = len(data)
    m = np.mean(data)
    se = stats.sem(data)
    t = stats.t.ppf(0.975, df=n-1)
    return m, m - t*se, m + t*se

---
## Pregunta 1
### ¿Cuántos sobres se necesitan en promedio para alcanzar distintos hitos del álbum (50%, 75%, 90%, 95% y 100%)?

**Motivación:** Completar el 100% es difícil por el efecto de las últimas estampas (coupon collector). Queremos saber cuándo el esfuerzo crece de manera desproporcionada.

In [ ]:
REPS_1 = 2000
HITOS = (0.50, 0.75, 0.90, 0.95, 1.00)

resultados_hitos = np.array([simular_album_hitos(hitos=HITOS) for _ in range(REPS_1)])

etiquetas = ['50 %', '75 %', '90 %', '95 %', '100 %']
medias   = resultados_hitos.mean(axis=0)
p5, p95  = np.percentile(resultados_hitos, 5, axis=0), np.percentile(resultados_hitos, 95, axis=0)

print(f"{'Hito':>6}  {'Media sobres':>14}  {'P5':>8}  {'P95':>8}  {'Costo medio (Q)':>16}")
print("-" * 60)
for lab, m, lo, hi in zip(etiquetas, medias, p5, p95):
    print(f"{lab:>6}  {m:>14.1f}  {lo:>8.0f}  {hi:>8.0f}  {m*P_SOBRE:>16,.2f}")

In [ ]:
# ── Gráfica 1a: barras con banda de error + incremento marginal ──────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x = np.arange(len(HITOS))
ax.bar(x, medias, color='steelblue', alpha=0.8, label='Media')
ax.errorbar(x, medias, yerr=[medias - p5, p95 - medias],
            fmt='none', color='black', capsize=5, label='P5–P95')
ax.set_xticks(x); ax.set_xticklabels(etiquetas)
ax.set_ylabel('Sobres'); ax.set_title('Sobres necesarios por hito')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))

ax2 = axes[1]
incremento = np.diff(medias)
etiq_inc = ['50→75 %', '75→90 %', '90→95 %', '95→100 %']
colores_inc = ['#4CAF50','#FFC107','#FF5722','#9C27B0']
ax2.bar(etiq_inc, incremento, color=colores_inc, alpha=0.85)
ax2.set_ylabel('Sobres adicionales')
ax2.set_title('Incremento marginal entre hitos')
for i, v in enumerate(incremento):
    ax2.text(i, v + 10, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('p1a_hitos_barras.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 1b: violin plots — distribución completa en cada hito ────
fig, ax = plt.subplots(figsize=(11, 5))

partes = ax.violinplot(
    [resultados_hitos[:, i] for i in range(len(HITOS))],
    positions=range(len(HITOS)),
    showmedians=True, showextrema=True
)
colores_violin = ['#4e9af1','#43aa8b','#f9c74f','#f8961e','#e63946']
for pc, color in zip(partes['bodies'], colores_violin):
    pc.set_facecolor(color)
    pc.set_alpha(0.75)
partes['cmedians'].set_color('black')
partes['cbars'].set_color('gray')
partes['cmaxes'].set_color('gray')
partes['cmins'].set_color('gray')

ax.set_xticks(range(len(HITOS)))
ax.set_xticklabels(etiquetas)
ax.set_ylabel('Sobres necesarios')
ax.set_title('Distribución de sobres por hito (violin plot)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
ax.grid(axis='y', alpha=0.3)

leyenda = [mpatches.Patch(color=c, alpha=0.75, label=l)
           for c, l in zip(colores_violin, etiquetas)]
ax.legend(handles=leyenda, loc='upper left')

plt.tight_layout()
plt.savefig('p1b_hitos_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 1c: curva de llenado promedio (± 1 std) ─────────────────
REPS_CURVA = 300
MAX_SOB = 1400

curvas = []
for _ in range(REPS_CURVA):
    c = simular_curva_llenado(max_sobres=MAX_SOB)
    # Rellenar hasta MAX_SOB con el valor final (álbum completo = N)
    if len(c) < MAX_SOB:
        c = np.append(c, np.full(MAX_SOB - len(c), N))
    curvas.append(c)

curvas = np.array(curvas)            # (REPS_CURVA, MAX_SOB)
media_curva = curvas.mean(axis=0)
std_curva   = curvas.std(axis=0)
x_curva     = np.arange(1, MAX_SOB + 1)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x_curva, media_curva, color='steelblue', lw=2, label='Media')
ax.fill_between(x_curva,
                np.clip(media_curva - std_curva, 0, N),
                np.clip(media_curva + std_curva, 0, N),
                alpha=0.25, color='steelblue', label='±1 desv. estándar')

# Líneas de hito
hito_colores = ['#43aa8b','#f9c74f','#f8961e','#e63946','#9C27B0']
for h, col, lab in zip(HITOS, hito_colores, etiquetas):
    ax.axhline(h * N, color=col, lw=1.5, linestyle='--', label=f'Hito {lab}')

ax.set_xlabel('Sobres abiertos')
ax.set_ylabel('Estampas únicas acumuladas')
ax.set_title('Curva de llenado del álbum (N = 980, S = 7)')
ax.set_xlim(0, MAX_SOB)
ax.set_ylim(0, N + 20)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('p1c_curva_llenado.png', bbox_inches='tight')
plt.show()

### Interpretación — Pregunta 1

El álbum se llena de manera **asimétrica**: los primeros dos tercios (hasta el 75%) se consiguen comprando relativamente pocos sobres, pero el salto del 95% al 100% requiere casi **tantos sobres como todo el tramo anterior**. Este es el efecto clásico del *coupon collector*: las últimas estampas son extremadamente difíciles de obtener porque la probabilidad de que un sobre traiga algo nuevo cae drásticamente.

Los **violin plots** revelan además que la variabilidad crece fuertemente en los hitos finales: en el 100% la distribución tiene una cola derecha muy larga (algunas simulaciones requieren el doble de sobres que la mediana).

La **curva de llenado** muestra que el crecimiento es casi lineal al principio y se aplana severamente a partir del 80%, confirmando visualmente el cuello de botella.

> **Resultado sorprendente:** pasar del 95% al 100% cuesta, en promedio, más sobres que pasar del 50% al 90%.

---
## Pregunta 2
### Con un presupuesto fijo de Q 500 comprando solo sobres individuales, ¿cuál es la distribución del porcentaje del álbum completado y la probabilidad de superar el 70%?

**Motivación:** Muchos coleccionistas no persiguen el 100%, sino un objetivo razonable. ¿Qué se puede lograr con Q 500?

In [ ]:
PRESUPUESTO = 500.0
SOBRES_DISPONIBLES = int(PRESUPUESTO // P_SOBRE)   # = 52
REPS_2 = 5000

print(f"Con Q{PRESUPUESTO:.0f} se pueden comprar {SOBRES_DISPONIBLES} sobres individuales.")

def simular_presupuesto(n=N, s=S, sobres_max=SOBRES_DISPONIBLES, rng=RNG):
    coleccion = np.zeros(n, dtype=bool)
    for _ in range(sobres_max):
        coleccion[rng.integers(0, n, size=s)] = True
    return coleccion.sum() / n

pct_album = np.array([simular_presupuesto() for _ in range(REPS_2)])

media_pct = pct_album.mean()
prob_70   = (pct_album >= 0.70).mean()

print(f"\nPorcentaje medio del álbum completado : {media_pct*100:.2f} %")
print(f"Prob. de superar el 70 %              : {prob_70*100:.2f} %")
print(f"Percentil 10 / 90                     : {np.percentile(pct_album,10)*100:.1f} % / {np.percentile(pct_album,90)*100:.1f} %")

In [ ]:
# ── Gráfica 2a: histograma de porcentaje completado ──────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(pct_album * 100, bins=40, color='teal', edgecolor='white', alpha=0.85)
ax.axvline(media_pct * 100, color='red', lw=2, label=f'Media: {media_pct*100:.1f} %')
ax.axvline(70, color='orange', lw=2, linestyle='--', label='Umbral 70 %')
ax.set_xlabel('% del álbum completado')
ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribución del álbum completado con Q{PRESUPUESTO:.0f} ({SOBRES_DISPONIBLES} sobres)')
ax.legend()
plt.tight_layout()
plt.savefig('p2a_presupuesto_hist.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 2b: CDF — P(% completado ≥ umbral) ──────────────────────
umbrales = np.linspace(0, 1, 300)
prob_superar = [(pct_album >= u).mean() for u in umbrales]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(umbrales * 100, prob_superar, color='teal', lw=2.5)
ax.axvline(70, color='orange', linestyle='--', lw=1.8, label='Umbral 70 %')
ax.axvline(media_pct * 100, color='red', linestyle='--', lw=1.8, label=f'Media {media_pct*100:.1f} %')
for u, col in [(0.30,'#4CAF50'),(0.40,'#8BC34A'),(0.50,'#FFC107')]:
    p = (pct_album >= u).mean()
    ax.plot(u*100, p, 'o', color=col, markersize=8)
    ax.annotate(f'{p*100:.0f}%', (u*100+0.5, p+0.01), fontsize=9, color=col)
ax.set_xlabel('Umbral de % del álbum')
ax.set_ylabel('P(completado ≥ umbral)')
ax.set_title('CDF complementaria: P(album% ≥ x)')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(0, 100); ax.set_ylim(0, 1.05)

# ── Gráfica 2c: comparar presupuestos ───────────────────────────────
ax2 = axes[1]
presupuestos = [250, 500, 1000, 2000, 5000]
colores_pres = ['#e63946','#f8961e','#f9c74f','#43aa8b','#4361ee']
for pres, col in zip(presupuestos, colores_pres):
    n_sob = int(pres // P_SOBRE)
    sim = np.array([simular_presupuesto(sobres_max=n_sob) for _ in range(2000)])
    ax2.hist(sim * 100, bins=30, alpha=0.55, color=col,
             label=f'Q{pres:,} ({n_sob} sobres)', density=True)
ax2.set_xlabel('% del álbum completado')
ax2.set_ylabel('Densidad')
ax2.set_title('Distribución por presupuesto')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('p2b_cdf_presupuestos.png', bbox_inches='tight')
plt.show()

### Interpretación — Pregunta 2

Con Q 500 (52 sobres = 364 estampas) se logra en promedio completar alrededor del **30–35%** del álbum. La probabilidad de superar el 70% con ese presupuesto es muy baja.

La **CDF complementaria** muestra cómo la probabilidad de alcanzar un umbral cae drásticamente: superar el 50% ya es improbable con Q 500. La comparación por presupuesto revela que se necesitan al menos **Q 5,000** para tener una distribución centrada cerca del 100%.

> **Reflexión:** Q 500 puede parecer mucho dinero, pero frente a un álbum de 980 estampas es una fracción pequeña del costo total esperado.

---
## Pregunta 3
### ¿Es más económico completar el álbum comprando cajas (Q 975/104 sobres) o sobres individuales (Q 9.50 c/u)? ¿Cuánto se ahorra en promedio comprando cajas?

**Motivación:** Las cajas son más baratas por sobre (Q 9.375 vs Q 9.50), pero deben comprarse en múltiplos de 104. ¿Compensa la obligación de comprar en lotes?

In [ ]:
P_SOBRE_CAJA = P_CAJA / SOBRES_CAJA   # Q 9.375 por sobre en caja
REPS_3 = 2000

def costo_suelto(sobres):
    return sobres * P_SOBRE

def costo_caja(sobres):
    cajas = int(np.ceil(sobres / SOBRES_CAJA))
    return cajas * P_CAJA

sobres_sim = np.array([simular_album() for _ in range(REPS_3)])

costos_sueltos = costo_suelto(sobres_sim)
costos_cajas   = np.array([costo_caja(s) for s in sobres_sim])
ahorro         = costos_sueltos - costos_cajas

print(f"Sobres promedio para completar el álbum : {sobres_sim.mean():.1f}")
print(f"Costo promedio — sobres sueltos          : Q {costos_sueltos.mean():,.2f}")
print(f"Costo promedio — comprando cajas         : Q {costos_cajas.mean():,.2f}")
print(f"Ahorro promedio (cajas vs. sueltos)      : Q {ahorro.mean():,.2f}")
print(f"% simulaciones donde cajas es más barato : {(ahorro > 0).mean()*100:.1f} %")

In [ ]:
# ── Gráfica 3a: histogramas superpuestos + histograma del ahorro ─────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
bins = np.linspace(min(costos_sueltos.min(), costos_cajas.min()),
                   max(costos_sueltos.max(), costos_cajas.max()), 40)
ax.hist(costos_sueltos, bins=bins, alpha=0.6, label='Sobres sueltos', color='steelblue')
ax.hist(costos_cajas,   bins=bins, alpha=0.6, label='Cajas',          color='coral')
ax.axvline(costos_sueltos.mean(), color='steelblue', lw=2, linestyle='--',
           label=f'Media sueltos: Q{costos_sueltos.mean():,.0f}')
ax.axvline(costos_cajas.mean(),   color='firebrick',  lw=2, linestyle='--',
           label=f'Media cajas: Q{costos_cajas.mean():,.0f}')
ax.set_xlabel('Costo total (Q)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución del costo para completar el álbum')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Q{int(v):,}'))

ax2 = axes[1]
ax2.hist(ahorro, bins=40, color='green', alpha=0.75, edgecolor='white')
ax2.axvline(ahorro.mean(), color='darkgreen', lw=2, label=f'Media: Q{ahorro.mean():,.0f}')
ax2.axvline(0, color='red', lw=1.5, linestyle='--', label='Sin ahorro')
ax2.set_xlabel('Ahorro al comprar cajas (Q)')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Ahorro de cajas vs. sobres sueltos')
ax2.legend()

plt.tight_layout()
plt.savefig('p3a_hist_costos.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 3b: scatter sueltos vs. cajas + box plots ───────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: un punto por simulación, coloreado según quién gana
ax = axes[0]
mask_caja_gana   = costos_cajas < costos_sueltos
mask_suelto_gana = ~mask_caja_gana
ax.scatter(costos_sueltos[mask_caja_gana],   costos_cajas[mask_caja_gana],
           color='coral', alpha=0.4, s=12, label='Caja más barata')
ax.scatter(costos_sueltos[mask_suelto_gana], costos_cajas[mask_suelto_gana],
           color='steelblue', alpha=0.4, s=12, label='Sueltos más baratos')
# Diagonal de igualdad
lim_min = min(costos_sueltos.min(), costos_cajas.min())
lim_max = max(costos_sueltos.max(), costos_cajas.max())
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', lw=1.5, label='Costo igual')
ax.set_xlabel('Costo sobres sueltos (Q)')
ax.set_ylabel('Costo cajas (Q)')
ax.set_title('Costo por simulación: sueltos vs. cajas')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Q{int(v):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Q{int(v):,}'))

# Box plots comparativos
ax2 = axes[1]
bp = ax2.boxplot([costos_sueltos, costos_cajas],
                 labels=['Sobres\nsueltos', 'Cajas'],
                 patch_artist=True, notch=True,
                 medianprops=dict(color='black', lw=2))
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('coral')
bp['boxes'][1].set_alpha(0.7)
ax2.set_ylabel('Costo total (Q)')
ax2.set_title('Box plot: costo total por estrategia')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Q{int(v):,}'))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('p3b_scatter_boxplot.png', bbox_inches='tight')
plt.show()

### Interpretación — Pregunta 3

Aunque el precio por sobre en caja es menor (Q 9.375 vs Q 9.50), la obligación de comprar múltiplos de 104 introduce **sobre-compra**: casi siempre se pagan sobres que no se necesitan.

El **scatter plot** es revelador: la mayoría de los puntos caen *por encima* de la diagonal, lo que significa que las cajas suelen ser más caras. El **box plot** con muesca confirma que la mediana de ambas distribuciones es muy cercana, pero la varianza de las cajas es mayor por los saltos discretos.

> **Reflexión:** La ventaja de la caja (precio por sobre) se ve anulada por el granulado de compra. Solo vale la pena si se va a comprar casi exactamente un múltiplo de 104 sobres.

---
## Pregunta 4
### Con una tasa de intercambio K = 6 (cada 6 repetidas se cambian por 1 nueva), ¿cuánto dinero se ahorra en promedio respecto a no intercambiar?

**Motivación:** El intercambio con amigos es una estrategia popular. Queremos cuantificar su impacto económico para el caso del álbum real.

In [ ]:
REPS_4 = 1500
K_VALS = [3, 4, 5, 6, 8, 10]

sobres_base = np.array([simular_album() for _ in range(REPS_4)])
costo_base  = sobres_base.mean() * P_SOBRE

resultados_k = {}
for k in K_VALS:
    sobres_k = np.array([simular_con_intercambio(k=k) for _ in range(REPS_4)])
    resultados_k[k] = sobres_k

print(f"Sin intercambio — sobres promedio: {sobres_base.mean():.1f}  |  costo: Q {costo_base:,.2f}")
print()
print(f"{'K':>4}  {'Sobres (media)':>16}  {'Costo medio (Q)':>16}  {'Ahorro (Q)':>12}  {'Ahorro (%)':>10}")
print("-" * 65)
for k in K_VALS:
    m = resultados_k[k].mean()
    costo_k = m * P_SOBRE
    ahorro_k = costo_base - costo_k
    print(f"{k:>4}  {m:>16.1f}  {costo_k:>16,.2f}  {ahorro_k:>12,.2f}  {ahorro_k/costo_base*100:>9.1f}%")

In [ ]:
# ── Gráfica 4a: línea de medias + barras de ahorro ───────────────────
medias_k  = [resultados_k[k].mean() for k in K_VALS]
ahorros_q = [costo_base - m * P_SOBRE for m in medias_k]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(K_VALS, medias_k, 'o-', color='steelblue', lw=2, markersize=8)
ax.axhline(sobres_base.mean(), color='red', linestyle='--', lw=1.8,
           label=f'Sin intercambio ({sobres_base.mean():.0f} sobres)')
ax.set_xlabel('Tasa de intercambio K')
ax.set_ylabel('Sobres promedio para completar')
ax.set_title('Efecto de K sobre sobres necesarios')
ax.legend(); ax.set_xticks(K_VALS); ax.grid(alpha=0.3)

ax2 = axes[1]
colores_k = ['#43aa8b','#90be6d','#f9c74f','#f8961e','#f3722c','#e63946']
bars = ax2.bar([str(k) for k in K_VALS], ahorros_q, color=colores_k, edgecolor='white')
ax2.set_xlabel('Tasa de intercambio K')
ax2.set_ylabel('Ahorro promedio (Q)')
ax2.set_title('Ahorro económico según tasa K')
for i, v in enumerate(ahorros_q):
    ax2.text(i, v + 20, f'Q{v:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('p4a_intercambio_linea.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 4b: violin plots — distribución de sobres por K ─────────
fig, ax = plt.subplots(figsize=(12, 5))

datos_violin = [sobres_base] + [resultados_k[k] for k in K_VALS]
etiq_violin  = ['Sin\nintercambio'] + [f'K = {k}' for k in K_VALS]
colores_viol = ['#e63946'] + colores_k

partes = ax.violinplot(datos_violin, positions=range(len(datos_violin)),
                       showmedians=True, showextrema=True)
for pc, color in zip(partes['bodies'], colores_viol):
    pc.set_facecolor(color)
    pc.set_alpha(0.72)
partes['cmedians'].set_color('black')
partes['cbars'].set_color('gray')
partes['cmaxes'].set_color('gray')
partes['cmins'].set_color('gray')

ax.set_xticks(range(len(etiq_violin)))
ax.set_xticklabels(etiq_violin)
ax.set_ylabel('Sobres necesarios para completar el álbum')
ax.set_title('Distribución de sobres según tasa de intercambio K')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
ax.grid(axis='y', alpha=0.3)

leyenda = [mpatches.Patch(color=c, alpha=0.72, label=l)
           for c, l in zip(colores_viol, etiq_violin)]
ax.legend(handles=leyenda, fontsize=9, ncol=4, loc='upper right')

plt.tight_layout()
plt.savefig('p4b_violin_K.png', bbox_inches='tight')
plt.show()

### Interpretación — Pregunta 4

El intercambio puede reducir significativamente el costo si la tasa K es baja. Con K = 3 el ahorro es considerable, mientras que con K = 10 el beneficio se vuelve marginal.

Los **violin plots** añaden una dimensión importante: con K bajo no solo se necesitan menos sobres en promedio, sino que la **variabilidad también se reduce** — la distribución es más estrecha y simétrica. Con K alto, la distribución se parece mucho al caso sin intercambio.

> **Reflexión:** El intercambio solo vale la pena económicamente si K ≤ 5. Con K = 6 el ahorro ya es modesto. Los violines muestran que K bajo también reduce la incertidumbre del costo, no solo el promedio.

---
## Pregunta 5
### ¿Cómo varía el número esperado de sobres para completar el álbum si S (estampas por sobre) cambia entre 5, 6, 7, 8 y 9?

**Motivación:** El fabricante podría decidir cambiar el número de estampas por sobre. ¿Cuánto impacto tiene esta decisión sobre el coleccionista?

In [ ]:
REPS_5 = 2000
S_VALS = [5, 6, 7, 8, 9]

res_s = {}
for s_val in S_VALS:
    sobres_s = np.array([simular_album(s=s_val) for _ in range(REPS_5)])
    res_s[s_val] = sobres_s

print(f"{'S':>4}  {'Sobres (media)':>16}  {'Estampas compradas':>20}  {'Costo (Q)':>12}  {'IC 95% inf':>12}  {'IC 95% sup':>12}")
print("-" * 85)
for s_val in S_VALS:
    arr = res_s[s_val]
    m, lo, hi = ci95(arr)
    print(f"{s_val:>4}  {m:>16.1f}  {m*s_val:>20.0f}  {m*P_SOBRE:>12,.2f}  {lo:>12.1f}  {hi:>12.1f}")

In [ ]:
# ── Gráfica 5a: línea + banda + barras de costo ──────────────────────
medias_s   = [res_s[s_val].mean()              for s_val in S_VALS]
p10_s      = [np.percentile(res_s[s_val], 10) for s_val in S_VALS]
p90_s      = [np.percentile(res_s[s_val], 90) for s_val in S_VALS]
costos_s   = [m * P_SOBRE                      for m in medias_s]
estampas_s = [m * s_val for m, s_val in zip(medias_s, S_VALS)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ax = axes[0]
ax.plot(S_VALS, medias_s, 'o-', color='steelblue', lw=2, markersize=9)
ax.fill_between(S_VALS, p10_s, p90_s, alpha=0.2, color='steelblue', label='P10–P90')
ax.set_xlabel('Estampas por sobre (S)')
ax.set_ylabel('Sobres necesarios')
ax.set_title('Sobres para completar el álbum')
ax.legend(); ax.set_xticks(S_VALS); ax.grid(alpha=0.3)

ax2 = axes[1]
colores_s = ['#e63946','#f8961e','#f9c74f','#43aa8b','#4361ee']
ax2.bar([str(s) for s in S_VALS], costos_s, color=colores_s, edgecolor='white')
ax2.set_xlabel('Estampas por sobre (S)')
ax2.set_ylabel('Costo promedio (Q)')
ax2.set_title('Costo para completar el álbum')
for i, v in enumerate(costos_s):
    ax2.text(i, v + 100, f'Q{v:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'Q{int(v):,}'))
ax2.grid(axis='y', alpha=0.3)

ax3 = axes[2]
ax3.plot(S_VALS, estampas_s, 's--', color='purple', lw=2, markersize=9)
ax3.axhline(N, color='gray', linestyle=':', lw=1.5, label='N = 980 (mínimo teórico)')
ax3.set_xlabel('Estampas por sobre (S)')
ax3.set_ylabel('Estampas totales compradas')
ax3.set_title('Total de estampas adquiridas (incluye repetidas)')
ax3.legend(); ax3.set_xticks(S_VALS); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('p5a_variacion_s.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Gráfica 5b: box plots + curvas de llenado por S ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plots
ax = axes[0]
bp = ax.boxplot([res_s[s_val] for s_val in S_VALS],
                labels=[f'S = {s}' for s in S_VALS],
                patch_artist=True, notch=True,
                medianprops=dict(color='black', lw=2))
for box, color in zip(bp['boxes'], colores_s):
    box.set_facecolor(color)
    box.set_alpha(0.75)
ax.set_ylabel('Sobres necesarios')
ax.set_title('Distribución de sobres por S (box plot)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{int(v):,}'))
ax.grid(axis='y', alpha=0.3)

# Curvas de llenado por S
ax2 = axes[1]
MAX_SOB_S = 1500
REPS_CURVA_S = 150
for s_val, color in zip(S_VALS, colores_s):
    curvas_s = []
    for _ in range(REPS_CURVA_S):
        c = simular_curva_llenado(s=s_val, max_sobres=MAX_SOB_S)
        if len(c) < MAX_SOB_S:
            c = np.append(c, np.full(MAX_SOB_S - len(c), N))
        curvas_s.append(c)
    media_c = np.array(curvas_s).mean(axis=0)
    ax2.plot(np.arange(1, MAX_SOB_S + 1), media_c,
             color=color, lw=2, label=f'S = {s_val}')

ax2.axhline(N, color='gray', linestyle=':', lw=1.5, label='Álbum completo')
ax2.set_xlabel('Sobres abiertos')
ax2.set_ylabel('Estampas únicas acumuladas')
ax2.set_title('Curva de llenado promedio por S')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
ax2.set_xlim(0, MAX_SOB_S); ax2.set_ylim(0, N + 20)

plt.tight_layout()
plt.savefig('p5b_boxplot_curvas_s.png', bbox_inches='tight')
plt.show()

### Interpretación — Pregunta 5

A medida que S aumenta, se necesitan **menos sobres** para completar el álbum, pero el total de estampas adquiridas (incluyendo repetidas) sube. El costo también baja porque se compran menos sobres.

Los **box plots** muestran que S más alto no solo reduce la media sino también la dispersión: los bigotes se acortan, lo que significa que el proceso es más predecible. Las **curvas de llenado** revelan que la diferencia entre S = 5 y S = 9 es especialmente marcada en la fase final del álbum (últimas 100 estampas).

> **Reflexión sorprendente:** Un sobre con 9 estampas no cuesta la mitad que uno con 5, pero el álbum se completa con significativamente menos sobres. Desde el punto de vista del coleccionista, sobres más grandes son siempre preferibles en costo y certeza.

---
## Conclusiones generales

| # | Pregunta | Hallazgo principal |
|---|---|---|
| 1 | Hitos del álbum | El tramo 95%→100% es tan costoso como el 50%→90% completo (efecto coupon collector); los violines confirman que la variabilidad también explota al final |
| 2 | Presupuesto Q500 | Con ese presupuesto (52 sobres) solo se completa ~30–35% del álbum; la CDF muestra que se necesitan al menos Q 5,000 para acercarse al 100% |
| 3 | Cajas vs. sueltos | El scatter muestra que la mayoría de puntos están sobre la diagonal: las cajas son más caras por la sobre-compra forzada |
| 4 | Intercambio K=6 | Los violin plots revelan que K bajo reduce tanto la media como la varianza del costo; solo conviene con K ≤ 5 |
| 5 | Variación de S | Box plots y curvas de llenado confirman que S mayor reduce media y dispersión; el efecto es más notorio en las últimas estampas |

**Reflexión final:** Completar un álbum de 980 estampas es extremadamente costoso sin estrategias complementarias (intercambio activo, compra de faltantes). La simulación confirma que el proceso tiene una naturaleza fuertemente asimétrica: las últimas estampas son las más caras de conseguir, y las visualizaciones adicionales demuestran que la variabilidad del costo crece junto con el promedio.